# 00-Setup: 10분 사전 점검

이 노트북은 본 수업에 들어가기 전에 먼저 진행하는 10분 사전 점검입니다. `01`~`06` 노트북에서는 동일한 고객 요청("결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.")을 여러 에이전트 아키텍처로 처리하며 각 구조를 선택한 근거를 비교합니다. 본격적인 실습에 앞서 내 컴퓨터 환경에서 다음 네 가지 계층이 정상적으로 동작하는지 차례대로 확인해 보겠습니다.

1. 모델을 호출하지 않는 LangGraph 기본 그래프
2. 실제 LLM 모델 단건 호출
3. 도구 하나를 연결한 Deep Agent 한 턴 실행
4. EXA 웹 검색 호출

## 학습 목표

- 패키지 설치는 저장소 루트에서 한 번만 수행하고, 실습 실행은 `Day03_new/` 폴더에서 수행해야 하는 이유를 이해합니다.
- 이 과정에 필요한 두 가지 API 키(`ANTHROPIC_API_KEY`, `EXA_API_KEY`)의 용도와 과금 주체를 확인합니다.
- 모델 없는 그래프, 실제 모델, Deep Agent, EXA 웹 검색 네 계층이 내 환경에서 정상적으로 동작하는지 출력으로 확인하고, 문제 발생 시 어느 계층의 오류인지 구분합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 설치와 폴더 구조 | 2 |
| 2 | 두 키 상태와 `.env` | 2 |
| 3 | 모델 없는 그래프와 사례 데이터 | 2 |
| 4 | 실제 모델, Deep Agent, EXA | 3 |
| 5 | 이후 시간표와 비용 | 1 |

## 호출 규모

모델을 실제로 호출하는 셀은 두 개입니다(모델 직접 호출 1회, Deep Agent 한 턴 실행). Deep Agent는 도구를 호출한 뒤 결과를 바탕으로 한 번 더 모델을 호출하므로 두 셀을 합쳐 실제 모델 호출은 약 3회이며, 셀마다 예고하는 상한 합계는 4회입니다. EXA 웹 검색은 1회 호출합니다. 앞쪽의 그래프 확인 및 데이터 조회 셀은 모델 호출 0회, EXA 호출 0회로 동작합니다. 유료 호출 비용은 본인의 Anthropic 크레딧과 EXA 크레딧에서 차감됩니다. 만약 API 키가 설정되어 있지 않다면 2절의 키 확인 셀에서 한국어 안내 메시지와 함께 즉시 멈추므로 유료 호출이 발생하지 않습니다.

## 1. 설치와 폴더 구조 (2분)

패키지 설치는 **저장소 루트에서 한 번만** 진행합니다. 루트의 `pyproject.toml`이 `uv` 워크스페이스 전체를 관리하며, `uv.lock`과 가상환경(`.venv`)도 루트에 하나만 존재합니다. `Day03_new/` 폴더에는 별도의 `pyproject.toml`이 없으며, `uv run` 실행 시 상위 폴더로 올라가 루트 가상환경을 자동으로 찾아 사용합니다.

```bash
uv sync --locked            # 저장소 루트에서 한 번만 실행합니다
cd Day03_new
uv run jupyter lab          # 실습 실행은 Day03_new 폴더에서 진행합니다
```

`Day03_new/` 폴더 안에서 `uv sync`를 실행하지 않도록 주의하세요. 현재 폴더에 선언되지 않은 패키지가 가상환경에서 제거되어 다른 Day 실습 환경이 깨질 수 있습니다. 만약 실수로 실행했다면 저장소 루트에서 `uv sync --locked`를 다시 실행해 복구할 수 있습니다.

실행 위치가 중요한 이유는 노트북 파일 옆에 위치한 `rag_lab/`, `mas_lab/` 패키지를 불러오기 위해서입니다. 커널(kernel)의 현재 작업 폴더가 `Day03_new/`여야 `import mas_lab`이 정상적으로 동작합니다.

아래 두 코드 셀에서는 현재 커널이 루트 `.venv` 환경을 사용하고 있는지, 그리고 두 패키지 폴더와 실습용 가상 데이터가 제자리에 있는지 확인합니다(모델 호출 0회, EXA 호출 0회).

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

# 현재 Python 버전과 주요 프레임워크 패키지 버전을 출력합니다.
print("Python            :", sys.version.split()[0])
for package in ("deepagents", "langgraph", "langchain-anthropic"):
    print(f"{package:<18}:", version(package))

# 가상환경 경로(.venv)와 현재 커널의 작업 폴더 위치를 확인합니다.
print("sys.prefix        :", sys.prefix)
print("루트 .venv에서 실행 중인가:", sys.prefix.endswith(".venv"))
print("kernel 작업 폴더  :", Path.cwd().name)

In [ ]:
import importlib
from collections import Counter

from mas_lab.view import show_table
from rag_lab.corpus import load_cases, load_corpus
from rag_lab.settings import ROOT

# 사내 원문 코퍼스와 평가 사례 데이터를 불러옵니다.
documents, cases = load_corpus(), load_cases()

# mas_lab 하위 모듈들이 오류 없이 import되는지 점검합니다.
modules = sorted(p.stem for p in (ROOT / "mas_lab").glob("*.py") if p.stem != "__init__")
broken = []
for name in modules:
    try:
        importlib.import_module(f"mas_lab.{name}")
    except Exception as error:
        broken.append(f"{name}: {type(error).__name__}")

# 고객 지원용 가상 사실 파일(결제 내역, 가용률·장애 기록) 존재 여부를 확인합니다.
facts = {name: (ROOT / "data" / "support" / name).exists() for name in ("billing.json", "outage.json")}
show_table([
    {"항목": "ROOT 폴더", "값": ROOT.name},
    {"항목": "원문 문서", "값": f"{len(documents)}개, 고객(entity)별 " + ", ".join(f"{k} {v}" for k, v in Counter(d.entity for d in documents.values()).items())},
    {"항목": "평가 사례 수", "값": f"{len(cases)}개"},
    {"항목": "사례 유형", "값": ", ".join(f"{kind} {count}" for kind, count in Counter(c["type"] for c in cases).items())},
    {"항목": "mas_lab 모듈", "값": f"{len(modules)}개 중 import 실패 {len(broken)}개 {broken}"},
    {"항목": "고정 사실 파일", "값": str(facts)},
])

## 2. 두 키 상태와 `.env` (2분)

이 과정에서는 두 가지 API 키를 사용합니다.

| 키 | 사용하는 곳 | 발급처 | 과금 주체 |
|---|---|---|---|
| `ANTHROPIC_API_KEY` | 모든 노트북의 LLM 모델 호출 | Anthropic 콘솔 | 본인 Anthropic 크레딧 |
| `EXA_API_KEY` | `06` 필수 리서치 실습 및 현재 노트북의 EXA 연동 확인 | `https://exa.ai/` (구글 로그인 권장) | 본인 EXA 크레딧 |

두 키는 반드시 `Day03_new/.env` 파일에 설정합니다. `Day03_new/.env.example` 양식을 복사한 뒤 본인의 키를 직접 입력해 주세요.

```bash
cp Day03_new/.env.example Day03_new/.env   # 복사 후 편집기로 두 API 키를 직접 입력합니다
```

만약 `Day02_new/.env` 파일을 이미 작성해 두었다면, 해당 파일을 복사한 뒤 `EXA_API_KEY` 한 줄만 추가해도 됩니다. 모델 설정 변수 이름이 `DAY02_MODEL`, `DAY02_EFFORT`로 되어 있는 이유도 `Day02_new/.env` 파일을 수정 없이 그대로 재사용할 수 있도록 하기 위함입니다.

- **보안 원칙**: API 키 원문은 노트북 출력, 로그, 결과 파일 어디에도 노출하지 않습니다. 상태 확인 함수는 오직 `설정됨`과 `미설정` 여부만 표시합니다.
- **환경 격리**: 셸(shell)에 `export`해 둔 환경 변수나 다른 Day 폴더의 `.env` 파일은 읽지 않습니다. 의도치 않게 다른 계정의 크레딧이 차감되는 사고를 방지하기 위함입니다.
- **커널 재시작**: `.env` 파일을 수정한 뒤에는 **커널을 재시작(Restart Kernel)**해야 변경 사항이 반영됩니다.
- **실제 모델 전용**: 이 과정은 가짜(mock) 대체 경로 없이 실제 모델로만 실행됩니다. 키가 없으면 키 확인 셀에서 즉시 멈춥니다.

아래 첫 번째 코드 셀에서는 API 키 설정 상태를 표로 확인한 뒤 두 키를 차례대로 검증하고 모델 객체를 생성합니다(객체 생성만 수행하며 API 호출은 하지 않습니다). 이어서 두 번째 코드 셀에서는 빈 설정을 넣어 키가 누락되었을 때 어떤 안내 메시지가 출력되는지 재현해 봅니다.

In [ ]:
from mas_lab.view import show_table
from rag_lab.settings import key_status, make_model, require_exa_key, require_key

# .env 파일의 키 설정 상태(설정됨/미설정)를 표로 출력합니다.
show_table([{"항목": name, "상태": status} for name, status in key_status().items()])

# 두 필수 API 키가 설정되어 있는지 검증합니다(누락 시 한국어 안내와 함께 RuntimeError 발생).
settings = require_key()
require_exa_key()

# ChatAnthropic 모델 객체를 초기화합니다(아직 네트워크 호출은 발생하지 않습니다).
model = make_model(settings)
print("모델 객체를 생성했습니다(API 호출 없음):", model.model, "| 추론 깊이(effort):", settings.effort)

In [ ]:
# 실패 재현: 빈 설정으로 두 키 검증 함수를 호출했을 때 출력되는 한국어 안내 메시지를 확인합니다
from rag_lab.settings import Settings

# 메모리 안에서만 빈 설정 객체를 생성합니다(실제 .env 파일은 변경하지 않습니다).
empty = Settings(api_key="")
for check in (require_key, require_exa_key):
    try:
        check(empty)
    except RuntimeError as error:
        print(f"{check.__name__:<16} -> {type(error).__name__}: {error}")

**관찰할 것.** 상태 표에서 `ANTHROPIC_API_KEY`와 `EXA_API_KEY`가 모두 `설정됨`으로 표시되는지 확인해 보세요. 하나라도 `미설정`이면 첫 번째 셀에서 실행이 멈추며, 멈춘 위치가 `require_key()`인지 `require_exa_key()`인지에 따라 어느 키를 채워야 하는지 바로 알 수 있습니다. 실패 재현 셀의 출력에서는 세 가지를 확인합니다. 발생한 예외가 `RuntimeError`인지, 안내 문구가 수정해야 할 파일과 변수 이름을 정확히 알려 주는지, 그리고 오류 메시지 어디에도 실제 키 값이 노출되지 않는지 살펴보세요. 이 안내가 출력된 시점에는 유료 API 호출이 한 번도 일어나지 않은 상태입니다.

## 3. 모델 없는 그래프와 사례 데이터 (2분)

이 과정의 많은 셀은 LLM 모델을 호출하지 않습니다. 그래프의 제어 구조, 상태 병합 리듀서(reducer), 쓰기 게이트, 멱등성(idempotency), 실패 분기 경로는 모델의 추론 없이도 검증할 수 있는 결정적(deterministic) 로직이므로 일반 Python 함수 노드로 실행합니다. 이는 키가 없을 때 우회하는 대체 경로가 아니라, 그래프의 구조적 동작을 명확하게 분리해 확인하기 위한 설계이며 출력에 `"모델 호출 0회"`를 명시합니다. 반대로 실제 모델을 호출하는 셀은 실행 직전에 `announce` 함수로 예상 호출 규모를 먼저 알립니다.

가장 간단한 예제로 병렬 팬아웃(fan-out) 그래프를 만들어 실행해 보겠습니다. 노드 이름인 `billing`, `policy`, `outage`는 `01`~`06`의 관통 사례에서 독립적으로 조사하는 세 가지 근거(결제 내역, 환불·보상 규정, 가용률·장애 기록)를 나타냅니다. 여기서는 각 노드가 `time.sleep(0.3)`으로 0.3초씩 대기하도록 구성했습니다. LangGraph의 `Send` API가 세 노드를 한 단계에서 동시에 실행하고, `Trace`가 각 노드의 시작·종료 시각을 기록하면 `overlap_seconds`가 실제로 겹쳐서 실행된 시간을 계산합니다.

**먼저 예측하세요.** 동일한 그래프를 두 번 실행합니다. 첫 번째는 기본 병렬 설정이고, 두 번째는 `config={"max_concurrency": 1}`로 동시 실행 수를 1로 제한한 설정입니다. 각 실행에서 `전체 시간(초)`이 세 노드 대기 시간의 합(약 0.9초)과 단일 노드 대기 시간(약 0.3초) 중 어느 쪽에 가까울지, 그리고 `겹친 시간(초)`이 0일지 0보다 클지 실행 전에 먼저 예측해 보세요.

아래 코드에서는 `Send`와 `operator.add` 리듀서를 사용한 병렬 그래프를 정의하고 구조를 시각화합니다.

In [ ]:
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from mas_lab.trace import Trace, overlap_seconds, timeline
from mas_lab.view import show_graph


# 병렬 노드가 동시에 결과를 추가할 수 있도록 Annotated와 operator.add 리듀서를 지정합니다.
class FanState(TypedDict):
    names: list[str]
    done: Annotated[list[str], operator.add]


def build_fan(trace: Trace):
    """START에서 Send로 여러 work 노드를 동시에 팬아웃하는 그래프를 생성합니다."""
    def work(state: dict) -> dict:
        with trace.span(state["name"]):
            time.sleep(0.3)  # 작업 지연을 모사하는 0.3초 대기입니다(모델 호출 없음).
        return {"done": [state["name"]]}

    builder = StateGraph(FanState)
    builder.add_node("work", work)
    builder.add_conditional_edges(START, lambda s: [Send("work", {"name": n}) for n in s["names"]], ["work"])
    builder.add_edge("work", END)
    return builder.compile()


# 정의한 그래프의 노드와 간선 구조를 출력합니다.
show_graph(build_fan(Trace("정의 확인")))

In [ ]:
# 기본 병렬 실행과 동시성 제한(max_concurrency=1) 실행의 소요 시간 및 겹친 시간을 비교합니다.
modes = {"기본": {}, "max_concurrency=1": {"max_concurrency": 1}}
rows, spans = [], {}
for label, config in modes.items():
    trace = Trace(label)
    started = time.perf_counter()
    result = build_fan(trace).invoke({"names": ["billing", "policy", "outage"]}, config=config)
    rows.append({"실행": label, "전체 시간(초)": round(time.perf_counter() - started, 3),
                 "겹친 시간(초)": overlap_seconds(trace.spans), "끝난 노드": len(result["done"])})
    spans[label] = trace.spans

show_table(rows)
for label, items in spans.items():
    print(f"[{label}]")
    print(timeline(items))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
import json

from mas_lab.desk import load_billing, load_outage
from mas_lab.view import show_table
from rag_lab.settings import ROOT

# 실습용 가상 결제 데이터(billing.json)와 가용률·장애 데이터(outage.json)를 확인합니다.
print("안내:", json.loads((ROOT / "data" / "support" / "billing.json").read_text(encoding="utf-8"))["_note"])
show_table([{"결제 번호": p["payment_id"], "청구서": p["invoice_id"], "승인 시각": p["approved_at"], "금액": p["amount"],
             "기본 이용료분": p["base_fee_part"], "초과분": p["usage_fee_part"]} for p in load_billing("알파")])
show_table([{"월": month, "가용률(%)": fact["availability_percent"],
             "사건": ", ".join(i["incident_id"] for i in fact["incidents"]) or "없음"}
            for month, fact in load_outage("알파").items()])

# 존재하지 않는 고객(entity)을 조회했을 때 예외 대신 빈 결과가 반환되는지 확인합니다.
print("없는 고객을 조회하면:", load_billing("없는회사"), load_outage("없는회사"))

In [ ]:
import time

from rag_lab.retrieval import tokenize

# 한국어 형태소 분석기(Kiwi)의 첫 호출(사전 적재 포함) 시간과 재호출 시간을 비교합니다.
started = time.perf_counter()
tokens = tokenize("서비스 크레딧 신청 기한")
first = time.perf_counter() - started

started = time.perf_counter()
tokenize("서비스 크레딧 신청 기한")
again = time.perf_counter() - started

print("토큰화 결과:", tokens)
print(f"첫 호출 {first:.2f}초 / 같은 문장 재호출 {again:.4f}초")

**관찰할 것.** 지금까지 모델 호출과 EXA 호출은 한 번도 없었으므로 발생한 비용은 0원입니다. 위 네 셀의 출력 결과를 다음 기준으로 살펴보세요.

- **그래프 정의 셀**: 간선 표에서 `조건부` 열이 `True`인 간선을 확인합니다. `Send`가 몇 개의 노드로 갈라질지는 실행 시점의 상태(`names`)에 따라 결정되므로, 정적 그래프 정의에는 `work` 노드 하나만 표시됩니다. 실제 병렬 실행 여부는 정적 그림이 아니라 아래 실행 표와 타임라인 기록으로 확인합니다.
- **병렬 그래프 실행 표**: `겹친 시간(초)` 열을 확인합니다. `기본` 행이 0보다 크고 `max_concurrency=1` 행이 0이면 설정 하나로 실제 병렬 실행과 순차 실행이 갈린 것입니다. 문자 막대 타임라인에서 세 작업 구간이 가로로 나란히 겹쳐 있는지 확인해 보세요. 두 행 모두 `끝난 노드` 수가 3개로 같은 것은 병렬 여부가 결과 건수를 바꾸지 않고 지연 시간만 바꾼다는 뜻입니다.
- **결제 및 가용률 표**: 결제 표에서 동일한 `청구서` 번호가 두 번 등장하는 행을 찾아 승인 시각이 얼마나 가까운지 확인해 보세요. 가용률 표에서는 장애 `사건`이 기록된 달과 `없음`인 달을 구분해 봅니다. 마지막 줄에서 존재하지 않는 고객을 조회했을 때 오류가 아니라 빈 결과(`[]`, `{}`)가 반환되는 점은, 이후 노트북에서 "데이터가 조회되지 않음"을 근거로 답해야 하는 사례와 이어집니다.
- **형태소 분석기(Kiwi) 호출 시간**: 첫 호출과 재호출 시간의 차이를 비교합니다. 첫 호출이 더 걸렸다면 그 차이는 형태소 사전 적재(ingest) 시간이며, 이미 앞 셀에서 적재되었다면 두 시간이 모두 짧게 나타납니다.

## 4. 실제 모델, Deep Agent, EXA (3분)

이제 유료 API를 사용하는 세 계층을 하나씩 점검합니다. 모든 유료 셀은 실행 직전에 `announce(...)`로 예상 호출 규모를 한 줄로 먼저 알립니다. 고지된 숫자는 해당 셀의 예상 상한이며, 실제 호출 수는 실행 후 출력에서 다시 확인합니다.

1. **모델 직접 호출**: `ANTHROPIC_API_KEY`가 유효한지, 모델이 정상 응답하는지, 응답 메타데이터가 어떻게 구성되는지 확인합니다.
2. **Deep Agent 한 턴 실행**: `create_deep_agent`에 결제 조회 도구 하나를 연결합니다. 도구의 고객(entity)과 기준일(as_of) 범위는 `Desk` 객체에 고정되어 있으므로 모델이 임의로 다른 고객을 조회할 수 없습니다. 실행 후 대화 기록(`messages`)이 몇 줄로 남는지 표로 확인합니다.
3. **EXA 웹 검색**: 웹 검색 1회가 검색 결과 제목과 URL을 정상적으로 반환하는지 확인합니다.

**먼저 예측하세요.** Deep Agent 셀을 실행하기 전에 메시지 표가 총 몇 줄로 구성될지, `종류` 열이 어떤 순서(`Human`, `AI`, `Tool`)로 나타날지, 그리고 모델 호출 집계(`model_calls`)가 몇 회일지 먼저 예측해 보세요. 도구를 1회 호출할 때 모델이 최소 몇 번 호출되는지도 함께 생각해 봅니다.

아래 세 코드 셀에서는 모델 직접 호출, Deep Agent 한 턴, EXA 웹 검색을 차례대로 실행합니다.

In [ ]:
from rag_lab.settings import announce

# 1. LLM 모델을 직접 1회 호출하여 응답과 토큰 사용량 메타데이터를 확인합니다.
announce("모델 직접 호출 확인", model_calls=1)
reply = model.invoke("한 단어로 답하세요: 준비")
usage = reply.usage_metadata or {}
print("모델 응답            :", reply.text)
print("내용 블록 종류       :", [b["type"] if isinstance(b, dict) else "text" for b in reply.content] if isinstance(reply.content, list) else "text")
print("요청한 모델 / 응답한 모델:", settings.model, "/", reply.response_metadata.get("model"))
print("입력 토큰 / 출력 토큰:", usage.get("input_tokens"), "/", usage.get("output_tokens"))

In [ ]:
from deepagents import create_deep_agent

from mas_lab.runner import make_desk
from mas_lab.trace import ModelMeter
from mas_lab.view import message_table
from rag_lab.corpus import load_cases

# 2. 관통 사례(complex-double-pay)용 Desk와 모델 호출 계측기(ModelMeter)를 준비합니다.
case = next(c for c in load_cases() if c["id"] == "complex-double-pay")
desk, meter = make_desk(case), ModelMeter()  # 고객(entity)과 기준일(as_of)은 desk에 고정됩니다.

announce("Deep Agent 한 턴", model_calls=3)
agent = create_deep_agent(model=model, tools=desk.tools("billing"), system_prompt="결제 사실은 도구로 조회한 값만 말하세요.")
question = "결제 내역을 조회해서 건수와, 같은 청구서에 겹친 결제가 있는지 한 줄로 알려 주세요."
result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"callbacks": [meter]})

# 에이전트가 주고받은 메시지 흐름과 모델 호출 집계, 등록된 사실 번호(fact_ref)를 확인합니다.
show_table(message_table(result["messages"]))
print("모델 호출 집계:", meter.totals())
print("조회로 등록된 사실 번호:", sorted(desk.fact_refs))

In [ ]:
from mas_lab.exa import ExaClient

# 3. EXA 웹 검색을 1회 호출하여 검색 결과 건수, 소요 시간, 비용, 출처 URL을 확인합니다.
announce("EXA 검색 확인", exa_calls=1)
client = ExaClient(max_searches=1, num_results=3, max_chars=300)
found = client.search("LangGraph Send API")
print(f"결과 {len(found)}건 | 검색 {client.searches}회 | {client.seconds:.2f}초 | 비용(달러) {client.cost_dollars}")
for row in found:
    print(row["source_id"], "|", row["title"][:50], "|", row["url"])

**관찰할 것.** 세 셀의 실행 결과를 각 열 이름으로 대조해 봅니다.

- **모델 직접 호출**: `내용 블록 종류`에 `text`만 있는지, 혹은 추론(`thinking`) 블록이 함께 포함되는지 확인합니다. `reply.text`는 텍스트 블록만 추출해 보여 주므로 블록이 여러 개여도 응답 본문을 깔끔하게 읽을 수 있습니다. `요청한 모델`과 `응답한 모델`이 일치하는지도 함께 확인하세요.
- **Deep Agent 메시지 표**: 표의 열은 `순번`, `종류`, `도구`, `인자`, `내용`입니다. `종류` 열의 순서(`Human` → `AI` → `Tool` → `AI`)가 예측과 일치하는지 확인해 보세요. 첫 번째 `AI` 행의 `도구` 열에 `lookup_payments`가 찍혀 있다면 모델이 스스로 도구 호출을 선택한 것입니다. 이때 `인자` 열을 보면 고객(entity) 인자가 없습니다. 고객 범위는 코드(`Desk`)가 고정하므로 모델이 임의로 바꿀 수 없기 때문입니다. 또한 `모델 호출 집계`의 `model_calls`가 `AI` 메시지 행의 개수와 일치하는지 확인해 보세요.
- **EXA 웹 검색**: 첫 줄의 `결과 N건`, 소요 시간, `비용(달러)`를 확인하고, 출력된 제목과 URL이 검색 질의(`LangGraph Send API`)와 부합하는지 살펴봅니다. 외부 웹 문서는 신뢰할 수 있는 명령이 아니라 검증 대상 자료라는 점이 `06` 실습의 출발점이 됩니다.

### 문제 발생 시 점검표

| 증상 | 먼저 확인할 곳 |
|---|---|
| 한국어 키 안내와 함께 `RuntimeError`가 발생함 | `Day03_new/.env` 파일 존재 여부, 변수 이름 철자, 키 입력 후 커널 재시작 여부 |
| `401 Unauthorized` 오류 | API 키 오타, 만료 여부, 복사 시 앞뒤 공백 포함 여부 확인 (모델 셀은 Anthropic, 검색 셀은 EXA 콘솔에서 확인) |
| `429 Too Many Requests` 오류 | 일시적인 호출 한도 초과입니다. 이 과정은 자동 재시도를 끄고 있으므로 약 1분 뒤 해당 셀부터 다시 실행합니다 |
| 시간 초과(Timeout) | 네트워크 연결 상태를 확인합니다 (모델 호출 제한 120초, EXA 호출 제한 45초) |
| `ModuleNotFoundError`, `ImportError` | 저장소 루트에서 `uv sync --locked` 실행 여부, 1절 출력에서 커널이 루트 `.venv`인지 및 작업 폴더가 `Day03_new`인지 확인 |

## 5. 이후 시간표와 비용 규칙 (1분)

| 노트북 | 주제 | 시간 |
|---|---|---:|
| `01` | Workflow와 Agent의 경계 | 60분 |
| `02` | Agent Architecture의 변화 | 60분 |
| `03a` / `03b` | Back to Basic: 네 가지 흐름 (순차·병렬 / 분기·루프) | 30분 + 30분 |
| `04a` / `04b` | Graph Engineering (계약·예산 / 실패·승인·멱등) | 30분 + 30분 |
| `05a` / `05b` | Multi-Agent 선택 기준 (실패·경계 / 측정·Fast/Slow·비교) | 30분 + 30분 |
| `06` | 리서치 Agent 실습 (EXA) | 60분 |

각 노트북의 첫 번째 셀 `호출 규모` 절에 해당 노트북의 예상 모델 호출 수와 EXA 검색 호출 수가 안내되어 있습니다. 모든 노트북은 앞 노트북의 커널 메모리 상태에 의존하지 않으므로 개별 노트북만 단독으로 열어도 실행됩니다.

실습 중 지켜야 할 비용 및 측정 규칙은 다섯 가지입니다.

1. **사전 규모 고지**: 유료 호출이 있는 셀은 실행 직전에 `announce(목적, model_calls=N, exa_calls=M)`로 예상 상한을 한 줄로 알립니다. 에이전트가 도구를 여러 번 호출하면 실제 호출 수가 달라질 수 있으므로 실행 후 출력 집계를 함께 확인합니다.
2. **결정적 구조는 모델 호출 0회**: 모델의 판단이 필요 없는 그래프 구조·리듀서·게이트 검증 셀은 모델을 호출하지 않고 출력에 `"모델 호출 0회"`를 표시합니다.
3. **선택 실행 스위치**: `직접 해보기`의 유료 셀은 첫 줄이 `RUN_EXERCISE = False`로 꺼져 있습니다. 값을 수정한 뒤 `True`로 바꿀 때만 실행되어 전체 실행 시 불필요한 비용이 추가되지 않도록 합니다. 규모가 큰 선택 비교 셀은 `RUN_OPTIONAL = False` 스위치를 사용합니다.
4. **1회 관찰 원칙**: 한 번 실행한 결과는 언제나 **"1회 관찰"**로 읽습니다. 동일 조건을 3회 이상 반복 측정하기 전에는 평균이나 구조 간 우열을 단정하지 않습니다.
5. **키 보안**: API 호출 비용은 본인의 크레딧에서 차감되며, API 키 원문은 화면·로그·결과 파일 어디에도 남기지 않습니다.

## 직접 해보기

### 해보기 1

`Day03_new/.env` 파일에서 `EXA_API_KEY=` 뒤의 값을 잠시 지우고 저장해 보세요(변수 이름 줄은 남겨 둡니다). 커널을 재시작한 뒤 위에서부터 순서대로 실행하면 어느 셀에서 어떤 안내 메시지와 함께 멈추는지 확인해 보세요. 실행 전에 멈추는 셀 위치와, 그 뒤에 있는 유료 모델 호출 셀들이 실행될지 먼저 예측해 본 뒤 실제 결과와 비교하고 키를 원래대로 복구합니다(키 값을 화면이나 노트북에 붙여 넣지 마세요).

### 해보기 2

작업 폴더가 `Day03_new/`가 아닐 때 `import mas_lab`이 어떻게 동작하는지 확인해 봅니다. 아래 코드 셀은 임시 폴더(`tempfile.gettempdir()`)에서 별도의 Python 프로세스를 띄워 `import mas_lab`을 시도합니다(모델 호출 0회). 먼저 그대로 실행해 종료 코드와 오류 마지막 줄을 확인한 뒤, `RUN_FOLDER`를 `str(ROOT)`로 바꾸어 다시 실행했을 때 결과가 어떻게 달라지는지 비교해 보세요.

In [ ]:
import os
import subprocess
import sys
import tempfile

from rag_lab.settings import ROOT

# 바꿀 값: tempfile.gettempdir() 대신 str(ROOT)로 바꾸어 다시 실행해 보세요.
RUN_FOLDER = tempfile.gettempdir()
no_color = {**os.environ, "PYTHON_COLORS": "0"}  # 오류 메시지에 ANSI 색상 제어 문자가 섞이지 않도록 설정합니다.

done = subprocess.run([sys.executable, "-c", "import mas_lab"], cwd=RUN_FOLDER, capture_output=True, text=True, env=no_color)
print("작업 폴더 :", RUN_FOLDER)
print("종료 코드 :", done.returncode)
print("마지막 줄 :", (done.stderr.strip().splitlines() or ["(오류 없음)"])[-1])

## 정리

- 패키지 설치는 저장소 루트에서 `uv sync --locked`로 한 번만 수행하고, 노트북 실행은 `Day03_new/` 폴더에서 진행합니다(`Day03_new/` 안에서 `uv sync`를 실행하지 않습니다).
- API 키는 `Day03_new/.env` 파일에 직접 설정합니다. `ANTHROPIC_API_KEY`는 모델 호출에, `EXA_API_KEY`는 웹 검색에 사용되며, 화면에는 `설정됨`과 `미설정` 상태만 표시됩니다.
- 이 과정은 실제 모델로만 동작합니다. 모델 판단이 필요 없는 그래프 구조 검증 셀은 모델 호출 0회로 결정적으로 실행하고, 유료 호출 셀은 `announce`로 규모를 먼저 알립니다.
- 모델 없는 그래프, 실제 모델 단건 호출, Deep Agent 한 턴, EXA 웹 검색 네 계층이 모두 정상 동작함을 확인했습니다.
- 다음 `01-Workflow-vs-Agent` 노트북에서는 "다음 행동을 누가 결정하는가"와 "실행 권한은 누구에게 있는가"를 기준으로 Workflow와 Agent의 경계를 살펴봅니다.

## 확인 질문

1. 3절의 병렬 그래프 셀과 4절의 Deep Agent 셀이 각각 검증하는 대상과 아직 검증하지 않는 대상은 무엇인가요?
2. 터미널 셸에 API 키를 `export`해 두어도 `key_status()`가 `미설정`으로 표시되는 이유는 무엇이며, 이 설계가 예방하려는 사고는 무엇인가요?